# Task A FC Neural Network

This notebook builds a fully connected neural network classifier for Task A (`RiskTier`).

Pipeline summary:
- Reuse the current Task A preprocessing from the tree-ensemble workflow
- Select the top 10 fold-local correlated processed features
- Add all 45 pairwise feature products
- Scale continuous features only
- Train an `MLPClassifier` with ReLU activations and the Adam optimizer
- Estimate performance with 5-fold stratified cross-validation
- Refit on the full training set and generate Task A test predictions

The final cell writes `taskA_nn.md` with the implementation notes and executed results.


In [1]:
from collections import Counter
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 5
TOP_K_FEATURES = 10
EXPECTED_BASE_FEATURES = 115
EXPECTED_INTERACTION_FEATURES = TOP_K_FEATURES * (TOP_K_FEATURES - 1) // 2
EXPECTED_TOTAL_FEATURES = EXPECTED_BASE_FEATURES + EXPECTED_INTERACTION_FEATURES

DATA_DIR = Path("creditsense-ai1215")
TRAIN_PATH = DATA_DIR / "credit_train.csv"
TEST_PATH = DATA_DIR / "credit_test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

NOTEBOOK_PATH = Path("taskA_nn.ipynb")
REPORT_PATH = Path("taskA_nn.md")
PREDICTION_PATH = Path("taskA_nn_risktier_predictions.csv")

CLASS_LABELS = [0, 1, 2, 3, 4]
CLASS_NAMES = ["VeryLow(0)", "Low(1)", "Moderate(2)", "High(3)", "VeryHigh(4)"]

print("Configuration loaded.")
print(f"Expected Task A base feature count: {EXPECTED_BASE_FEATURES}")
print(f"Expected engineered interaction count: {EXPECTED_INTERACTION_FEATURES}")
print(f"Expected final NN feature count: {EXPECTED_TOTAL_FEATURES}")


Configuration loaded.
Expected Task A base feature count: 115
Expected engineered interaction count: 45
Expected final NN feature count: 160


In [2]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

X = train_df.drop(columns=["RiskTier", "InterestRate"])
y = train_df["RiskTier"].astype(int)

print(f"Train rows/cols: {train_df.shape}")
print(f"Test rows/cols: {test_df.shape}")
print(f"Task A feature matrix: {X.shape}")
print("RiskTier distribution:")
print(y.value_counts().sort_index().to_string())


Train rows/cols: (35000, 57)
Test rows/cols: (15000, 55)
Task A feature matrix: (35000, 55)
RiskTier distribution:
RiskTier
0    6724
1    7283
2    6998
3    6812
4    7183


In [3]:
TASK_A_ZERO_FILL_COLS = [
    "StudentLoanOutstandingBalance",
    "MortgageOutstandingBalance",
    "PropertyValue",
    "InvestmentPortfolioValue",
    "VehicleValue",
    "AutoLoanOutstandingBalance",
    "SecondaryMonthlyIncome",
    "CollateralValue",
]

TASK_A_MONEY_CLIP_COLS = [
    "AnnualIncome",
    "MonthlyGrossIncome",
    "SecondaryMonthlyIncome",
    "TotalMonthlyIncome",
    "SavingsBalance",
    "CheckingBalance",
    "InvestmentPortfolioValue",
    "PropertyValue",
    "VehicleValue",
    "TotalAssets",
    "MortgageOutstandingBalance",
    "AutoLoanOutstandingBalance",
    "StudentLoanOutstandingBalance",
    "TotalCreditLimit",
    "RequestedLoanAmount",
    "CollateralValue",
    "MonthlyPaymentEstimate",
]

TASK_A_RATIO_CLIP_COLS = ["LoanToIncomeRatio", "PaymentToIncomeRatio"]
TASK_A_LOG_COLS = TASK_A_MONEY_CLIP_COLS.copy()


def _fit_tabular_preprocessor(
    X_train_raw: pd.DataFrame,
    *,
    zero_fill_cols: list[str],
    clip_cols: list[str],
    log_cols: list[str],
) -> dict:
    X_train_raw = X_train_raw.copy()

    base_cols = X_train_raw.columns.tolist()
    num_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X_train_raw.select_dtypes(include=["object", "category"]).columns.tolist()
    missing_flag_cols = [col for col in base_cols if X_train_raw[col].isna().any()]

    zero_fill_cols = [col for col in zero_fill_cols if col in num_cols]
    median_fill_map = {
        col: float(X_train_raw[col].median())
        for col in num_cols
        if col not in zero_fill_cols
    }

    clip_map = {}
    for col in clip_cols:
        if col not in num_cols:
            continue
        non_null = X_train_raw[col].dropna()
        clip_map[col] = float(non_null.quantile(0.99)) if not non_null.empty else 0.0

    prep = {
        "cat_cols": cat_cols,
        "num_cols": num_cols,
        "missing_flag_cols": missing_flag_cols,
        "zero_fill_cols": zero_fill_cols,
        "median_fill_map": median_fill_map,
        "clip_map": clip_map,
        "log_cols": [col for col in log_cols if col in num_cols],
    }

    transformed_train = _transform_tabular(X_train_raw, prep, align=False)
    prep["feature_columns"] = transformed_train.columns.tolist()
    return prep


def _transform_tabular(X_raw: pd.DataFrame, prep: dict, align: bool = True) -> pd.DataFrame:
    working = X_raw.copy()

    for col in prep["missing_flag_cols"]:
        working[f"{col}_is_missing"] = working[col].isna().astype(np.int8)

    for col in prep["cat_cols"]:
        working[col] = working[col].fillna("Missing").astype(str)

    for col in prep["zero_fill_cols"]:
        working[col] = working[col].fillna(0.0)

    for col, med in prep["median_fill_map"].items():
        working[col] = working[col].fillna(med)

    for col, upper in prep["clip_map"].items():
        working[col] = working[col].clip(upper=upper)

    for col in prep["log_cols"]:
        working[col] = np.log1p(working[col])

    numeric_frame = working[prep["num_cols"]].apply(pd.to_numeric)
    missing_cols = [f"{col}_is_missing" for col in prep["missing_flag_cols"]]
    missing_frame = (
        working[missing_cols].astype(np.int8)
        if missing_cols
        else pd.DataFrame(index=working.index)
    )
    cat_frame = pd.get_dummies(
        working[prep["cat_cols"]],
        prefix=prep["cat_cols"],
        prefix_sep="__",
        drop_first=False,
        dtype=np.int8,
    )

    transformed = pd.concat([numeric_frame, missing_frame, cat_frame], axis=1).fillna(0)
    if align and "feature_columns" in prep:
        transformed = transformed.reindex(columns=prep["feature_columns"], fill_value=0)
    return transformed


def fit_task_a_preprocessor(X_train_raw: pd.DataFrame) -> dict:
    return _fit_tabular_preprocessor(
        X_train_raw,
        zero_fill_cols=TASK_A_ZERO_FILL_COLS,
        clip_cols=TASK_A_MONEY_CLIP_COLS + TASK_A_RATIO_CLIP_COLS,
        log_cols=TASK_A_LOG_COLS,
    )


def transform_task_a(X_raw: pd.DataFrame, prep: dict) -> pd.DataFrame:
    transformed = _transform_tabular(X_raw, prep, align=True)
    return transformed.reindex(columns=prep["feature_columns"], fill_value=0)


In [4]:
def select_top_correlated_features(
    X_processed: pd.DataFrame,
    y_fold: pd.Series,
    prep: dict,
    top_k: int = TOP_K_FEATURES,
) -> pd.DataFrame:
    missing_indicator_cols = [
        f"{col}_is_missing"
        for col in prep["missing_flag_cols"]
        if f"{col}_is_missing" in X_processed.columns
    ]
    candidate_cols = [
        col for col in prep["num_cols"] + missing_indicator_cols
        if col in X_processed.columns
    ]
    corr = X_processed[candidate_cols].corrwith(y_fold).dropna()
    corr_df = (
        pd.DataFrame({
            "feature": corr.index,
            "correlation": corr.values,
        })
        .assign(abs_correlation=lambda df: df["correlation"].abs())
        .sort_values(["abs_correlation", "feature"], ascending=[False, True])
        .head(top_k)
        .reset_index(drop=True)
    )
    if len(corr_df) != top_k:
        raise ValueError(f"Expected {top_k} correlated features, got {len(corr_df)}")
    return corr_df


def add_pairwise_interactions(
    X_frame: pd.DataFrame,
    selected_features: list[str],
) -> tuple[pd.DataFrame, list[str]]:
    transformed = X_frame.copy()
    interaction_cols = []
    for i, left in enumerate(selected_features):
        for right in selected_features[i + 1:]:
            name = f"{left}__x__{right}"
            transformed[name] = transformed[left] * transformed[right]
            interaction_cols.append(name)
    return transformed, interaction_cols


def build_task_a_mlp() -> MLPClassifier:
    return MLPClassifier(
        hidden_layer_sizes=(256, 128),
        activation="relu",
        solver="adam",
        learning_rate_init=3e-4,
        alpha=1e-4,
        batch_size=256,
        max_iter=400,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-8,
        shuffle=True,
        random_state=RANDOM_STATE,
    )


def fit_task_a_nn_pipeline(
    X_train_raw: pd.DataFrame,
    y_train: pd.Series,
) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    prep = fit_task_a_preprocessor(X_train_raw)
    X_train_base = transform_task_a(X_train_raw, prep)

    if X_train_base.isnull().sum().sum() != 0:
        raise ValueError("Task A preprocessing left missing values in the training fold.")
    if X_train_base.shape[1] != EXPECTED_BASE_FEATURES:
        raise ValueError(
            f"Expected {EXPECTED_BASE_FEATURES} Task A features, got {X_train_base.shape[1]}"
        )

    top_corr_df = select_top_correlated_features(X_train_base, y_train, prep, top_k=TOP_K_FEATURES)
    selected_features = top_corr_df["feature"].tolist()

    X_train_nn, interaction_cols = add_pairwise_interactions(X_train_base, selected_features)
    if len(interaction_cols) != EXPECTED_INTERACTION_FEATURES:
        raise ValueError(
            f"Expected {EXPECTED_INTERACTION_FEATURES} interactions, got {len(interaction_cols)}"
        )

    continuous_cols = [col for col in prep["num_cols"] if col in X_train_nn.columns] + interaction_cols
    scaler = StandardScaler()
    X_train_nn.loc[:, continuous_cols] = scaler.fit_transform(X_train_nn[continuous_cols])

    if X_train_nn.isnull().sum().sum() != 0:
        raise ValueError("Scaling introduced missing values in the training fold.")

    feature_columns = X_train_nn.columns.tolist()
    if len(feature_columns) != EXPECTED_TOTAL_FEATURES:
        raise ValueError(
            f"Expected {EXPECTED_TOTAL_FEATURES} total features, got {len(feature_columns)}"
        )

    X_train_final = X_train_nn.reindex(columns=feature_columns, fill_value=0).astype(np.float32)

    state = {
        "prep": prep,
        "selected_features": selected_features,
        "top_corr_df": top_corr_df.copy(),
        "interaction_cols": interaction_cols,
        "continuous_cols": continuous_cols,
        "scaler": scaler,
        "feature_columns": feature_columns,
    }
    return state, X_train_final, top_corr_df


def transform_task_a_nn(X_raw: pd.DataFrame, state: dict) -> pd.DataFrame:
    X_base = transform_task_a(X_raw, state["prep"])
    X_nn, interaction_cols = add_pairwise_interactions(X_base, state["selected_features"])

    if interaction_cols != state["interaction_cols"]:
        raise ValueError("Interaction columns drifted between fit and transform.")

    X_nn.loc[:, state["continuous_cols"]] = state["scaler"].transform(X_nn[state["continuous_cols"]])
    X_final = X_nn.reindex(columns=state["feature_columns"], fill_value=0).astype(np.float32)

    if X_final.isnull().sum().sum() != 0:
        raise ValueError("Task A NN transform left missing values behind.")
    if X_final.shape[1] != EXPECTED_TOTAL_FEATURES:
        raise ValueError(
            f"Expected {EXPECTED_TOTAL_FEATURES} total features, got {X_final.shape[1]}"
        )
    return X_final


def format_value(value, float_digits: int = 4) -> str:
    if pd.isna(value):
        return ""
    if isinstance(value, (np.integer, int)):
        return str(int(value))
    if isinstance(value, (np.floating, float)):
        value = float(value)
        if abs(value - round(value)) < 1e-12:
            return str(int(round(value)))
        return f"{value:.{float_digits}f}"
    return str(value)


def dataframe_to_markdown(df: pd.DataFrame, index: bool = False, float_digits: int = 4) -> str:
    working = df.copy()
    if index:
        index_name = working.index.name or "index"
        working = working.reset_index().rename(columns={"index": index_name})

    columns = [str(col) for col in working.columns]
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for row in working.itertuples(index=False, name=None):
        rows.append("| " + " | ".join(format_value(value, float_digits=float_digits) for value in row) + " |")
    return "\n".join([header, separator] + rows)


In [5]:
def run_task_a_nn_cv(X_raw: pd.DataFrame, y_raw: pd.Series) -> dict:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    y_array = y_raw.to_numpy()
    oof_pred = np.full(len(X_raw), -1, dtype=int)

    fold_records = []
    fold_feature_rows = []
    feature_counter = Counter()

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_raw, y_raw), start=1):
        X_train_raw = X_raw.iloc[train_idx].reset_index(drop=True)
        X_val_raw = X_raw.iloc[val_idx].reset_index(drop=True)
        y_train = y_raw.iloc[train_idx].reset_index(drop=True)
        y_val = y_raw.iloc[val_idx].reset_index(drop=True)

        state, X_train_final, top_corr_df = fit_task_a_nn_pipeline(X_train_raw, y_train)
        X_val_final = transform_task_a_nn(X_val_raw, state)

        if not X_train_final.columns.equals(X_val_final.columns):
            raise ValueError("Train/validation schemas do not match in a CV fold.")

        model = build_task_a_mlp()
        model.fit(X_train_final, y_train)
        val_pred = model.predict(X_val_final).astype(int)
        oof_pred[val_idx] = val_pred

        fold_accuracy = float(accuracy_score(y_val, val_pred))
        fold_macro_f1 = float(f1_score(y_val, val_pred, average="macro"))

        fold_records.append({
            "fold": fold_idx,
            "train_rows": len(train_idx),
            "val_rows": len(val_idx),
            "base_feature_count": EXPECTED_BASE_FEATURES,
            "interaction_count": len(state["interaction_cols"]),
            "final_feature_count": X_train_final.shape[1],
            "epochs": int(model.n_iter_),
            "accuracy": fold_accuracy,
            "macro_f1": fold_macro_f1,
        })

        feature_counter.update(state["selected_features"])
        for rank, row in enumerate(top_corr_df.itertuples(index=False), start=1):
            fold_feature_rows.append({
                "fold": fold_idx,
                "rank": rank,
                "feature": row.feature,
                "correlation": float(row.correlation),
                "abs_correlation": float(row.abs_correlation),
            })

        print(
            f"Fold {fold_idx}: "
            f"accuracy={fold_accuracy:.4f}, "
            f"macro_f1={fold_macro_f1:.4f}, "
            f"epochs={int(model.n_iter_)}, "
            f"top_features={state['selected_features']}"
        )

    if (oof_pred < 0).any():
        raise ValueError("OOF predictions do not cover every training row exactly once.")

    fold_metrics_df = pd.DataFrame(fold_records)
    feature_frequency_df = (
        pd.DataFrame(
            [{"feature": feature, "count": count} for feature, count in feature_counter.items()]
        )
        .sort_values(["count", "feature"], ascending=[False, True])
        .reset_index(drop=True)
    )
    fold_features_df = pd.DataFrame(fold_feature_rows)

    confusion_df = pd.DataFrame(
        confusion_matrix(y_array, oof_pred, labels=CLASS_LABELS),
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    )
    confusion_df.index.name = "actual"

    classification_df = pd.DataFrame(
        classification_report(
            y_array,
            oof_pred,
            labels=CLASS_LABELS,
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0,
        )
    ).T
    classification_df.index.name = "label"

    summary_df = pd.DataFrame([
        {
            "metric": "accuracy",
            "mean": float(fold_metrics_df["accuracy"].mean()),
            "std": float(fold_metrics_df["accuracy"].std(ddof=1)),
        },
        {
            "metric": "macro_f1",
            "mean": float(fold_metrics_df["macro_f1"].mean()),
            "std": float(fold_metrics_df["macro_f1"].std(ddof=1)),
        },
    ])

    return {
        "fold_metrics_df": fold_metrics_df,
        "summary_df": summary_df,
        "confusion_df": confusion_df,
        "classification_df": classification_df,
        "fold_features_df": fold_features_df,
        "feature_frequency_df": feature_frequency_df,
        "oof_pred": oof_pred,
    }


CV_RESULTS = run_task_a_nn_cv(X, y)

print("\nCross-validation metrics:")
print(CV_RESULTS["fold_metrics_df"].round(4).to_string(index=False))
print("\nMetric summary:")
print(CV_RESULTS["summary_df"].round(4).to_string(index=False))
print("\nTop feature frequency across folds:")
print(CV_RESULTS["feature_frequency_df"].to_string(index=False))
print("\nOut-of-fold confusion matrix:")
print(CV_RESULTS["confusion_df"].to_string())
print("\nOut-of-fold classification report:")
print(CV_RESULTS["classification_df"].round(4).to_string())


Fold 1: accuracy=0.8127, macro_f1=0.8142, epochs=38, top_features=['NumberOfLatePayments30Days', 'RevolvingUtilizationRate', 'NumberOfChargeOffs', 'NumberOfCollections', 'NumberOfLatePayments60Days', 'AnnualIncome', 'NumberOfBankruptcies', 'NumberOfLatePayments90Days', 'LoanToIncomeRatio', 'NumberOfHardInquiries12Mo']


Fold 2: accuracy=0.8240, macro_f1=0.8246, epochs=41, top_features=['NumberOfLatePayments30Days', 'RevolvingUtilizationRate', 'NumberOfChargeOffs', 'NumberOfCollections', 'NumberOfLatePayments60Days', 'AnnualIncome', 'NumberOfBankruptcies', 'NumberOfLatePayments90Days', 'LoanToIncomeRatio', 'NumberOfHardInquiries12Mo']


Fold 3: accuracy=0.8193, macro_f1=0.8207, epochs=38, top_features=['NumberOfLatePayments30Days', 'RevolvingUtilizationRate', 'NumberOfChargeOffs', 'NumberOfCollections', 'NumberOfLatePayments60Days', 'AnnualIncome', 'NumberOfBankruptcies', 'NumberOfLatePayments90Days', 'LoanToIncomeRatio', 'NumberOfHardInquiries12Mo']


Fold 4: accuracy=0.8159, macro_f1=0.8169, epochs=41, top_features=['NumberOfLatePayments30Days', 'RevolvingUtilizationRate', 'NumberOfChargeOffs', 'NumberOfCollections', 'NumberOfLatePayments60Days', 'AnnualIncome', 'NumberOfBankruptcies', 'NumberOfLatePayments90Days', 'LoanToIncomeRatio', 'NumberOfHardInquiries12Mo']


Fold 5: accuracy=0.8069, macro_f1=0.8067, epochs=47, top_features=['NumberOfLatePayments30Days', 'RevolvingUtilizationRate', 'NumberOfChargeOffs', 'NumberOfCollections', 'NumberOfLatePayments60Days', 'AnnualIncome', 'NumberOfBankruptcies', 'NumberOfLatePayments90Days', 'LoanToIncomeRatio', 'NumberOfHardInquiries12Mo']

Cross-validation metrics:
 fold  train_rows  val_rows  base_feature_count  interaction_count  final_feature_count  epochs  accuracy  macro_f1
    1       28000      7000                 115                 45                  160      38    0.8127    0.8142
    2       28000      7000                 115                 45                  160      41    0.8240    0.8246
    3       28000      7000                 115                 45                  160      38    0.8193    0.8207
    4       28000      7000                 115                 45                  160      41    0.8159    0.8169
    5       28000      7000                 115                 45       

In [6]:
def fit_full_model_and_predict_test(
    X_train_raw: pd.DataFrame,
    y_train: pd.Series,
    X_test_raw: pd.DataFrame,
    sample_submission_df: pd.DataFrame,
) -> dict:
    state, X_train_final, global_top_corr_df = fit_task_a_nn_pipeline(X_train_raw, y_train)
    model = build_task_a_mlp()
    model.fit(X_train_final, y_train)

    X_test_final = transform_task_a_nn(X_test_raw, state)
    test_pred = model.predict(X_test_final).astype(int)

    if len(test_pred) != len(sample_submission_df):
        raise ValueError("Prediction row count does not match sample submission.")
    if test_pred.min() < min(CLASS_LABELS) or test_pred.max() > max(CLASS_LABELS):
        raise ValueError("Task A predictions fell outside the expected [0, 4] range.")

    submission = sample_submission_df[["Id"]].copy()
    submission["RiskTier"] = test_pred
    submission.to_csv(PREDICTION_PATH, index=False)

    prediction_distribution_df = (
        pd.Series(test_pred, name="RiskTier")
        .value_counts()
        .sort_index()
        .rename_axis("RiskTier")
        .reset_index(name="count")
    )

    return {
        "state": state,
        "model": model,
        "global_top_corr_df": global_top_corr_df,
        "submission_df": submission,
        "prediction_distribution_df": prediction_distribution_df,
    }


FINAL_RESULTS = fit_full_model_and_predict_test(X, y, test_df, sample_submission)

print("Global top 10 correlated features used for final inference:")
print(FINAL_RESULTS["global_top_corr_df"].round(4).to_string(index=False))
print("\nPrediction distribution on test set:")
print(FINAL_RESULTS["prediction_distribution_df"].to_string(index=False))
print(f"\nSaved Task A predictions to {PREDICTION_PATH}")


Global top 10 correlated features used for final inference:
                   feature  correlation  abs_correlation
NumberOfLatePayments30Days       0.5707           0.5707
  RevolvingUtilizationRate       0.5432           0.5432
        NumberOfChargeOffs       0.4504           0.4504
       NumberOfCollections       0.4413           0.4413
NumberOfLatePayments60Days       0.4318           0.4318
              AnnualIncome      -0.3804           0.3804
      NumberOfBankruptcies       0.3690           0.3690
NumberOfLatePayments90Days       0.3215           0.3215
         LoanToIncomeRatio       0.2747           0.2747
 NumberOfHardInquiries12Mo       0.2561           0.2561

Prediction distribution on test set:
 RiskTier  count
        0   2629
        1   3656
        2   2704
        3   3170
        4   2841

Saved Task A predictions to taskA_nn_risktier_predictions.csv


In [7]:
fold_top10_summary_df = (
    CV_RESULTS["fold_features_df"]
    .sort_values(["fold", "rank"])
    .groupby("fold")["feature"]
    .apply(lambda items: ", ".join(items))
    .reset_index(name="top_10_features")
)

report_lines = [
    "# Task A FC Neural Network Report",
    "",
    "## Objective",
    "Implemented a new Task A classifier in `taskA_nn.ipynb` using a fully connected neural network with ReLU activations and the Adam optimizer. The notebook reuses the existing tree-ensemble preprocessing, adds fold-local engineered interaction features, evaluates the model with 5-fold stratified cross-validation, and generates Task A test predictions.",
    "",
    "## Dataset And Task Scope",
    f"- Training data: {len(train_df)} rows, {train_df.shape[1]} columns",
    f"- Test data: {len(test_df)} rows, {test_df.shape[1]} columns",
    f"- Task A features before preprocessing: {X.shape[1]}",
    f"- Classification target: `RiskTier` with labels {CLASS_LABELS}",
    "",
    "## Preprocessing Reused From The Tree Ensemble Model",
    "- Train-fold-only preprocessing fit and validation/test transform",
    "- Missing categorical values filled with the explicit `Missing` category",
    "- Full one-hot encoding for categorical variables",
    "- Numeric missing-value indicators added before imputation",
    "- Structural numeric missingness zero-filled for the Task A zero-fill columns",
    "- Remaining numeric features median-imputed with train-fold statistics only",
    "- 99th-percentile clipping applied to the Task A money and ratio columns",
    "- `log1p` applied to the Task A money columns",
    "- Validation and test matrices reindexed to the exact train-fold schema",
    "",
    "## Engineered Feature Design",
    f"- Within each training fold, the top {TOP_K_FEATURES} processed features were ranked by absolute Pearson correlation with `RiskTier`.",
    "- Candidate features were restricted to processed numeric columns plus `*_is_missing` indicator columns.",
    "- One-hot dummy columns were excluded from the correlation ranking.",
    f"- All {EXPECTED_INTERACTION_FEATURES} pairwise products were added as engineered features.",
    f"- Final NN design matrix size: {EXPECTED_BASE_FEATURES} base features + {EXPECTED_INTERACTION_FEATURES} interactions = {EXPECTED_TOTAL_FEATURES} total features.",
    "- Only the processed numeric columns and engineered interaction columns were standardized; one-hot columns and missing indicators remained as 0/1 inputs.",
    "",
    "## FC Network Architecture And Adam / Backprop Configuration",
    "- `MLPClassifier(hidden_layer_sizes=(256, 128), activation='relu', solver='adam')`",
    "- `learning_rate_init=3e-4`, `alpha=1e-4`, `batch_size=256`, `max_iter=400`",
    "- `early_stopping=True`, `validation_fraction=0.1`, `n_iter_no_change=20`",
    "- Adam moment parameters: `beta_1=0.9`, `beta_2=0.999`, `epsilon=1e-8`",
    "- `shuffle=True`, `random_state=42`",
    "",
    "Adam update summary used by the model:",
    "",
    "```text",
    "m_t = beta_1 * m_(t-1) + (1 - beta_1) * g_t",
    "v_t = beta_2 * v_(t-1) + (1 - beta_2) * (g_t ** 2)",
    "theta_(t+1) = theta_t - eta * m_hat_t / (sqrt(v_hat_t) + epsilon)",
    "```",
    "",
    "## 5-Fold Evaluation Protocol",
    f"- `StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={RANDOM_STATE})`",
    "- For each fold: fit preprocessing, select top correlated features, add pairwise products, scale continuous inputs, train the MLP, and score the held-out fold.",
    "- Aggregate all held-out predictions into one out-of-fold prediction vector for overall confusion-matrix and classification-report evaluation.",
    "",
    "## Executed Results",
    "### Fold Metrics",
    dataframe_to_markdown(CV_RESULTS["fold_metrics_df"], index=False, float_digits=4),
    "",
    "### Mean And Standard Deviation",
    dataframe_to_markdown(CV_RESULTS["summary_df"], index=False, float_digits=4),
    "",
    "### Out-Of-Fold Confusion Matrix",
    dataframe_to_markdown(CV_RESULTS["confusion_df"], index=True, float_digits=0),
    "",
    "### Out-Of-Fold Classification Report",
    dataframe_to_markdown(CV_RESULTS["classification_df"], index=True, float_digits=4),
    "",
    "### Fold-Wise Top 10 Correlated Features",
    dataframe_to_markdown(fold_top10_summary_df, index=False, float_digits=4),
    "",
    "### Detailed Fold Feature Ranking",
    dataframe_to_markdown(CV_RESULTS["fold_features_df"], index=False, float_digits=4),
    "",
    "### Feature Frequency Across Folds",
    dataframe_to_markdown(CV_RESULTS["feature_frequency_df"], index=False, float_digits=0),
    "",
    "### Global Top 10 Correlated Features Used For Final Inference",
    dataframe_to_markdown(FINAL_RESULTS["global_top_corr_df"], index=False, float_digits=4),
    "",
    "### Test Prediction Distribution",
    dataframe_to_markdown(FINAL_RESULTS["prediction_distribution_df"], index=False, float_digits=0),
    "",
    "## Generated Artifacts",
    f"- Notebook: `{NOTEBOOK_PATH.name}`",
    f"- Markdown report: `{REPORT_PATH.name}`",
    f"- Task A predictions: `{PREDICTION_PATH.name}`",
    f"- Prediction rows written: {len(FINAL_RESULTS['submission_df'])}",
]

REPORT_PATH.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

print(f"Saved markdown report to {REPORT_PATH}")
print(f"Saved prediction file to {PREDICTION_PATH}")


Saved markdown report to taskA_nn.md
Saved prediction file to taskA_nn_risktier_predictions.csv
